<a href="https://colab.research.google.com/github/RazyAnas/MachineLearning/blob/main/Column_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

In [3]:
from sklearn.impute import SimpleImputer # If some values are missing, replace them with a sensible guess with strategy='something'
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [4]:
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/100-days-of-machine-learning/refs/heads/main/day28-column-transformer/covid_toy.csv')

In [5]:
df.sample(5)

,age,gender,fever,cough,city,has_covid
40,49,Female,102.0,Mild,Delhi,No
26,19,Female,100.0,Mild,Kolkata,Yes
49,44,Male,104.0,Mild,Mumbai,No
24,13,Female,100.0,Strong,Kolkata,No
35,82,Female,102.0,Strong,Bangalore,No


In [6]:
df['city'].value_counts()

,count
city,
Kolkata,32
Bangalore,30
Delhi,22
Mumbai,16


In [7]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [ ]:
# Nominal Categorical Data: gender, city, has_covid
# Ordinal Categorical Data: age, fever, cough

In [9]:
# before any transformation first do train test split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']), df['has_covid'], test_size=0.2)

In [10]:
X_train

,age,gender,fever,cough,city
40,49,Female,102.0,Mild,Delhi
13,64,Male,102.0,Mild,Bangalore
45,72,Male,99.0,Mild,Bangalore
49,44,Male,104.0,Mild,Mumbai
46,19,Female,101.0,Mild,Mumbai
...,...,...,...,...,...
65,69,Female,102.0,Mild,Bangalore
1,27,Male,100.0,Mild,Delhi
82,24,Male,98.0,Mild,Kolkata
92,82,Female,102.0,Strong,Kolkata


# 1. Aam Zindagi

In [11]:
# adding simple imputer to fever col

In [12]:
si = SimpleImputer() # object of simple imputer, used mean

In [19]:
X_train_fever = si.fit_transform(X_train[['fever']])
X_test_fever = si.fit_transform(X_test[['fever']])
X_train_fever.shape

(80, 1)

In [16]:
# OrdinalEncoding -> cough # order

In [17]:
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

X_test_cough = oe.fit_transform(X_test[['cough']])
X_train_cough.shape

(80, 1)

In [21]:
# OneHotEncoding -> gender,city # no order
ohe = OneHotEncoder(drop='first',sparse_output=False)

X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])
X_test_gender_city = ohe.fit_transform(X_test[['gender', 'city']])

X_train_gender_city.shape

(80, 4)

In [23]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender', 'fever', 'cough', 'city']).values
# also the test data
X_test_age = X_test.drop(columns=['gender', 'fever', 'cough', 'city']).values
X_train_age.shape

(80, 1)

In [26]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever, X_train_gender_city, X_train_cough),axis=1)
X_test_transformed = np.concatenate((X_test_age,X_test_fever, X_test_gender_city, X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

# Mentos Zindagi

In [27]:
from sklearn.compose import ColumnTransformer

In [31]:
transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(), ['fever']),
    ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
    ('tnf3', OneHotEncoder(sparse_output=False, drop='first'), ['gender', 'city'])
], remainder='passthrough')

In [34]:
transformer.fit_transform(X_train).shape

(80, 7)

In [35]:
transformer.transform(X_test).shape

(20, 7)

# Practice

In [36]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("khushikyad001/india-road-accident-dataset-predictive-analysis")

print("Path to dataset files:", path)

100%|██████████| 68.2k/68.2k [00:00<00:00, 18.7MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/khushikyad001/india-road-accident-dataset-predictive-analysis/versions/1


In [37]:
df = pd.read_csv('/root/.cache/kagglehub/datasets/khushikyad001/india-road-accident-dataset-predictive-analysis/versions/1/accident_prediction_india.csv')

In [38]:
df.sample(5)

,State Name,City Name,Year,Month,Day of Week,Time of Day,Accident Severity,Number of Vehicles Involved,Vehicle Type Involved,Number of Casualties,...,Road Type,Road Condition,Lighting Conditions,Traffic Control Presence,Speed Limit (km/h),Driver Age,Driver Gender,Driver License Status,Alcohol Involvement,Accident Location Details
2776,Uttarakhand,Unknown,2023,March,Saturday,16:36,Serious,3,Auto-Rickshaw,10,...,National Highway,Under Construction,Dawn,Signs,62,24,Female,NaN,Yes,Straight Road
2333,Kerala,Unknown,2018,August,Wednesday,5:45,Serious,4,Bus,0,...,Urban Road,Under Construction,Dawn,Signals,82,35,Male,Valid,Yes,Bridge
2629,Goa,Unknown,2019,March,Saturday,19:5,Serious,4,Bus,5,...,Urban Road,Damaged,Daylight,NaN,76,20,Female,NaN,No,Straight Road
1345,Jammu and Kashmir,Unknown,2023,February,Monday,0:6,Serious,5,Auto-Rickshaw,2,...,Village Road,Wet,Dawn,Police Checkpost,52,50,Male,Expired,Yes,Bridge
1997,Madhya Pradesh,Unknown,2018,June,Thursday,14:58,Minor,5,Auto-Rickshaw,9,...,State Highway,Under Construction,Dawn,Signals,97,55,Female,Valid,No,Straight Road


In [39]:
df = df[['State Name', 'Day of Week', 'Accident Severity', 'Vehicle Type Involved', 'Number of Casualties', 'Number of Fatalities', 'Weather Conditions', 'Road Condition', 'Lighting Conditions', 'Traffic Control Presence', 'Speed Limit (km/h)', 'Driver Age', 'Driver Gender', 'Driver License Status', 'Alcohol Involvement']]

Index(['State Name', 'City Name', 'Year', 'Month', 'Day of Week',
       'Time of Day', 'Accident Severity', 'Number of Vehicles Involved',
       'Vehicle Type Involved', 'Number of Casualties', 'Number of Fatalities',
       'Weather Conditions', 'Road Type', 'Road Condition',
       'Lighting Conditions', 'Traffic Control Presence', 'Speed Limit (km/h)',
       'Driver Age', 'Driver Gender', 'Driver License Status',
       'Alcohol Involvement', 'Accident Location Details'],
      dtype='object')

In [42]:
df.isnull().sum()

,0
State Name,0
City Name,0
Year,0
Month,0
Day of Week,0
Time of Day,0
Accident Severity,0
Number of Vehicles Involved,0
Vehicle Type Involved,0
Number of Casualties,0


In [63]:
df['Number of Fatalities'].sample(10)

,Number of Fatalities
1337,3
2987,0
431,4
1457,5
1726,2
595,0
1576,1
1538,2
2913,4
317,1


In [59]:
# Nominal Categorical Data:
# (Categories with no natural order)
cols = ['Driver License Status']
nominal_cols = [
    'State Name',
    'Day of Week',
    'Weather Conditions',
    'Road Condition',
    'Lighting Conditions',
    'Traffic Control Presence',
    'Driver Gender',
    'Alcohol Involvement',
    'Vehicle Type Involved'
]


# Ordinal Categorical Data:
# (Categories that have a meaningful order/ranking)

ordinal_cols = [
    'Accident Severity',
]

In [64]:
# before any transformation first do train test split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Number of Fatalities']), df['Number of Fatalities'], test_size=0.2)

In [69]:
from sklearn.compose import ColumnTransformer


In [92]:
imputer = SimpleImputer(strategy='most_frequent')

X_train.loc[:, cols] = imputer.fit_transform(X_train[cols])
X_test.loc[:, cols] = imputer.transform(X_test[cols])

transformer = ColumnTransformer(transformers=[
    ('tnf1', OrdinalEncoder(categories=[['Expired', 'Valid']]), cols),
    ('tnf2', OrdinalEncoder(categories=[['Minor', 'Serious','Severe', 'Fatal']]), ordinal_cols),
    ('tnf3', OneHotEncoder(
        sparse_output=False,
        drop='first',
        handle_unknown='ignore'
    ), nominal_cols)
], remainder='drop')

X_train_transformed = transformer.fit_transform(X_train)
X_test_transformed = transformer.transform(X_test)

In [93]:
# Train the model
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train_transformed, y_train)

# Make predictions
y_pred = model.predict(X_test_transformed)

# Evaluate the model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("MAE :", mean_absolute_error(y_test, y_pred))
print("MSE :", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R²  :", r2_score(y_test, y_pred))

MAE : 1.5192724813111333
MSE : 3.0098609405906434
RMSE: 1.7348950805713421
R²  : -0.016879266390973813
